# HR Data Analysis – Exploratory Data Analysis (EDA)

Analysis of a 2,000,000-record HR dataset covering departments, job roles, experience, salary, performance, employee status and work mode.

**Notebook flow**
1. Setup & data loading
2. Data validation (types, duplicates, missing values, ID & date consistency)
3. Outlier detection & treatment (IQR)
4. Univariate analysis (headcount, work mode, status, job titles, hiring trend)
5. Bivariate analysis (salary by department / job title / experience, performance by department)
6. Correlation analysis
7. Insights

> **Dataset:** place `HR_Data_MNC_Data Science Lovers.csv` in the same folder as this notebook (see README).

In [2]:
%pip install matplotlib seaborn


  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.66.1-cp312-cp312-win_amd64.whl.metadata (132 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl (234 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.66.1-cp312-cp312-win_amd64.whl (2.5 MB)
Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl (70 kB)
Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl (7.2 MB)
U

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## 1. Setup & data loading

In [3]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 100
inr_fmt = FuncFormatter(lambda x, _: f"{x/1e6:.1f}M")   # axis labels like 0.8M instead of 1e6 offsets

DATA_PATH = "HR_Data_MNC_Data Science Lovers.csv"   # keep the CSV next to this notebook
df = pd.read_csv(DATA_PATH)
df.head()

Matplotlib is building the font cache; this may take a moment.


FileNotFoundError: [Errno 2] No such file or directory: 'HR_Data_MNC_Data Science Lovers.csv'

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Data validation & cleaning

### 2.1 Drop the redundant index column and check data types

In [ ]:
# 'Unnamed: 0' is just the old row index saved into the CSV
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
print("Shape:", df.shape)
df.dtypes

In [ ]:
# Hire_Date is stored as text -> convert to datetime
df['Hire_Date'] = pd.to_datetime(df['Hire_Date'], errors='coerce')
print("Hire_Date dtype:", df['Hire_Date'].dtype)
print("Dates that failed to parse:", df['Hire_Date'].isna().sum())

### 2.2 Missing values & duplicates

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nFully duplicated rows:", df.duplicated().sum())

### 2.3 Duplicate Employee IDs

In [ ]:
# Employee_ID must be unique - one row per employee
dup_ids = df['Employee_ID'].duplicated().sum()
print(f"Duplicate Employee_ID values: {dup_ids:,}")
print(f"Unique Employee_IDs: {df['Employee_ID'].nunique():,} of {len(df):,} rows")

if dup_ids > 0:
    display(df[df['Employee_ID'].duplicated(keep=False)].sort_values('Employee_ID').head(10))
else:
    print("OK - every row has a unique Employee_ID")

### 2.4 Hire-date consistency (future dates)

In [ ]:
today = pd.Timestamp.today().normalize()

future_hires = df[df['Hire_Date'] > today]
print(f"Hire dates in the future (after {today.date()}): {len(future_hires):,}")
print(f"Earliest hire date: {df['Hire_Date'].min().date()}")
print(f"Latest hire date  : {df['Hire_Date'].max().date()}")

if len(future_hires) > 0:
    display(future_hires.head())
else:
    print("OK - no hire dates are in the future")

In [ ]:
# Does stated total experience make sense next to time spent at the company?
# Total experience should be >= years since hire. Rows where it is smaller are inconsistent.
tenure_years = (today - df['Hire_Date']).dt.days / 365.25
inconsistent = df[df['Experience_Years'] < np.floor(tenure_years)]
print(f"Rows where Experience_Years < tenure at the company: {len(inconsistent):,} "
      f"({len(inconsistent) / len(df):.1%})")
print("Informational only - these rows are kept, but experience-based findings should be read with this in mind.")

### 2.5 Range and category checks

In [ ]:
print("Performance_Rating within 1-5   :", df['Performance_Rating'].between(1, 5).all())
print("Experience_Years >= 0           :", (df['Experience_Years'] >= 0).all())
print("Salary_INR > 0                  :", (df['Salary_INR'] > 0).all())
print()
for col in ['Department', 'Status', 'Work_Mode']:
    print(f"{col}: {sorted(df[col].unique())}")
print("\nUnique job titles:", df['Job_Title'].nunique())

## 3. Outlier detection & treatment (IQR method)

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.histplot(df['Salary_INR'], bins=30, kde=True)
plt.title("Salary distribution (before outlier treatment) - right-skewed")
plt.xlabel("Salary (INR)")
plt.show()

In [ ]:
q1, q3 = df['Salary_INR'].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

is_outlier = (df['Salary_INR'] < lower_bound) | (df['Salary_INR'] > upper_bound)
print(f"Q1 = {q1:,.0f} | Q3 = {q3:,.0f} | IQR = {iqr:,.0f}")
print(f"Valid range: {lower_bound:,.0f} to {upper_bound:,.0f}")
print(f"Outliers found: {is_outlier.sum():,} ({is_outlier.mean():.2%} of rows)")

In [ ]:
# Cap (winsorize) instead of dropping, so no employee rows are lost.
# The raw salary is kept in Salary_Original for reference.
df['Salary_Original'] = df['Salary_INR']
df['Salary_INR'] = df['Salary_INR'].clip(lower=lower_bound, upper=upper_bound)

remaining = ((df['Salary_INR'] < lower_bound) | (df['Salary_INR'] > upper_bound)).sum()
print("Outliers remaining after capping:", remaining)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
sns.boxplot(y=df['Salary_Original'], ax=axes[0], color='#e07a5f')
axes[0].set_title("Before capping")
sns.boxplot(y=df['Salary_INR'], ax=axes[1], color='#81b29a')
axes[1].set_title("After capping")
for ax in axes:
    ax.set_ylabel("Salary (INR)")
plt.suptitle("Salary outliers - IQR method")
plt.tight_layout()
plt.show()

## 4. Univariate analysis

### Which departments have the largest headcount?

In [ ]:
dept_counts = df['Department'].value_counts()
plt.figure(figsize=(9, 5))
ax = sns.barplot(x=dept_counts.index, y=dept_counts.values, color='skyblue')
ax.bar_label(ax.containers[0], fmt='{:,.0f}', padding=3)
plt.title("Employees by department")
plt.xlabel("Department")
plt.ylabel("Number of employees")
plt.show()

### What is the most common work mode?

In [ ]:
wm = df['Work_Mode'].value_counts()
plt.figure(figsize=(5.5, 5.5))
plt.pie(wm.values, labels=wm.index, autopct='%1.1f%%', startangle=90,
        colors=sns.color_palette('pastel'), explode=[0.03] * len(wm))
plt.title("Employee work mode")
plt.show()

### What is the current employee status mix?

In [ ]:
st = df['Status'].value_counts()
plt.figure(figsize=(5.5, 5.5))
plt.pie(st.values, labels=st.index, autopct='%1.1f%%', startangle=90,
        colors=sns.color_palette('Set2'), explode=[0.03] * len(st))
plt.title("Employee status")
plt.show()

### Are there job titles with very few employees compared to others?

In [ ]:
title_counts = df['Job_Title'].value_counts()
plt.figure(figsize=(9, 9))
sns.barplot(x=title_counts.values, y=title_counts.index, color='skyblue')
plt.title("Employees by job title")
plt.xlabel("Number of employees")
plt.ylabel("")
plt.show()

print("Smallest job titles:")
print(title_counts.tail(5))

### Which years had the most hiring?

In [ ]:
# Hires per calendar year (uses ALL rows, grouped by year)
hires_per_year = df['Hire_Date'].dt.year.value_counts().sort_index()
peak_year = hires_per_year.idxmax()

plt.figure(figsize=(10, 5))
plt.plot(hires_per_year.index, hires_per_year.values, marker='o', linewidth=2)
plt.scatter(peak_year, hires_per_year[peak_year], color='red', zorder=5, s=90,
            label=f"Peak: {peak_year} ({hires_per_year[peak_year]:,} hires)")
plt.title("Hires per year")
plt.xlabel("Hire year")
plt.ylabel("Number of employees hired")
plt.xticks(hires_per_year.index, rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# note: the last year may be partial if it is still in progress
hires_per_year

## 5. Bivariate analysis

### Which department pays the most on average?

In [ ]:
dept_salary = df.groupby('Department')['Salary_INR'].mean().sort_values(ascending=False)
plt.figure(figsize=(9, 5))
ax = sns.barplot(x=dept_salary.index, y=dept_salary.values, color='#81b29a')
ax.bar_label(ax.containers[0], labels=[f"{v/1e3:,.0f}K" for v in dept_salary.values], padding=3)
plt.title("Average salary by department")
plt.xlabel("Department")
plt.ylabel("Average salary (INR)")
ax.yaxis.set_major_formatter(inr_fmt)
plt.show()

### Average salary by job title (colored by department)

In [ ]:
job_salary = (df.groupby(['Department', 'Job_Title'])['Salary_INR']
                .mean().reset_index().sort_values('Salary_INR', ascending=False))

plt.figure(figsize=(10, 9))
sns.barplot(data=job_salary, x='Salary_INR', y='Job_Title', hue='Department', dodge=False)
plt.title("Average salary by job title")
plt.xlabel("Average salary (INR)")
plt.ylabel("")
plt.gca().xaxis.set_major_formatter(inr_fmt)
plt.legend(title="Department", loc='lower right')
plt.show()

job_salary.assign(Salary_K=(job_salary['Salary_INR'] / 1000).round(1)).drop(columns='Salary_INR').head(10)

### How does salary vary with years of experience?

In [ ]:
exp_salary = df.groupby('Experience_Years')['Salary_INR'].mean()
corr_exp_salary = df['Experience_Years'].corr(df['Salary_INR'])

plt.figure(figsize=(10, 5))
plt.plot(exp_salary.index, exp_salary.values, marker='o', linewidth=2, color='#3d405b')
plt.ylim(0, exp_salary.max() * 1.25)    # start at 0 so tiny fluctuations are not exaggerated
plt.title(f"Average salary vs. years of experience (correlation = {corr_exp_salary:.3f})")
plt.xlabel("Experience (years)")
plt.ylabel("Average salary (INR)")
plt.xticks(exp_salary.index)
plt.gca().yaxis.set_major_formatter(inr_fmt)
plt.show()

print(f"Lowest average : {exp_salary.min():,.0f} (at {exp_salary.idxmin()} yrs)")
print(f"Highest average: {exp_salary.max():,.0f} (at {exp_salary.idxmax()} yrs)")
print(f"Spread between them: {(exp_salary.max() - exp_salary.min()) / exp_salary.mean():.2%} of the mean")

### What is the average performance rating by department?

In [ ]:
perf = df.groupby('Department')['Performance_Rating'].mean().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=perf.index, y=perf.values, color='#f2cc8f')
ax.bar_label(ax.containers[0], fmt='%.3f', padding=3)
plt.ylim(0, 5)
plt.axhline(df['Performance_Rating'].mean(), color='grey', linestyle='--',
            label=f"Company average ({df['Performance_Rating'].mean():.2f})")
plt.title("Average performance rating by department (scale 1-5)")
plt.xlabel("Department")
plt.ylabel("Average rating")
plt.legend()
plt.show()

## 6. Correlation analysis

In [ ]:
num_cols = ['Performance_Rating', 'Experience_Years', 'Salary_INR']
plt.figure(figsize=(6, 5))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title("Correlation between numeric features")
plt.show()

## 7. Insights

### 7.1 Generated from the data (re-computed on every run)

In [ ]:
top_dept = dept_counts.idxmax()
top_dept_salary = dept_salary.idxmax()
best_job = job_salary.iloc[0]
worst_job = job_salary.iloc[-1]

print(f"1. Largest department: {top_dept} ({dept_counts.max():,} employees, {dept_counts.max()/len(df):.1%} of workforce).")
print(f"2. Most common work mode: {wm.idxmax()} ({wm.max()/len(df):.1%}).")
print(f"3. Employee status: " + ", ".join(f"{k} {v/len(df):.1%}" for k, v in st.items()) + ".")
print(f"4. Peak hiring year: {peak_year} ({hires_per_year.max():,} hires).")
print(f"5. Highest-paying department (avg): {top_dept_salary} (INR {dept_salary.max():,.0f}).")
print(f"6. Highest-paid job title (avg): {best_job['Job_Title']} - {best_job['Department']} (INR {best_job['Salary_INR']:,.0f}); "
      f"lowest: {worst_job['Job_Title']} - {worst_job['Department']} (INR {worst_job['Salary_INR']:,.0f}).")
print(f"7. Salary vs experience correlation: {corr_exp_salary:.3f}; "
      f"average salary varies only {(exp_salary.max()-exp_salary.min())/exp_salary.mean():.2%} across 0-15 years.")
print(f"8. Performance rating: department averages range {perf.min():.3f} to {perf.max():.3f} "
      f"(spread {perf.max()-perf.min():.3f} on a 1-5 scale).")
print(f"9. Salary outliers capped with IQR: {is_outlier.sum():,} rows ({is_outlier.mean():.2%}).")

### 7.2 Key takeaways

**Workforce structure**
- **IT is the largest department** (about 30% of employees), followed by Sales, Operations, Marketing and Finance.
- **60% of employees work On-site and 40% Remote.**
- **About 70% of employees are Active**; roughly 20% have Resigned, and Retired and Terminated are about 5% each.

**Compensation**
- **Job title drives pay, not experience.** Average salary is flat (~INR 0.88M) from 0 to 15 years of experience, and the experience-salary correlation is essentially zero.
- **Manager-level roles earn the most:** IT Manager (~INR 1.67M) and Finance Manager (~INR 1.56M) lead, followed by HR Manager and Business Development Manager (~INR 1.25M each) and Software Engineer (~INR 1.2M). The lowest-paid title is HR Executive (~INR 0.55M).
- **Unusual pattern:** C-level titles (CFO, CTO) average ~INR 0.8M, below several manager roles. That is unrealistic and suggests the dataset is synthetic.

**Performance**
- **Performance is uniform across departments:** every department averages about 3.0 out of 5, with differences of roughly 0.01. No department stands out as high or low performing.

**Data quality**
- No missing values, no duplicate rows, and the data types are consistent after converting `Hire_Date` to datetime. About 3.5% of salaries were flagged as IQR outliers and capped rather than removed.
- The Employee_ID uniqueness and future-hire-date checks (sections 2.3 and 2.4) print their results directly above; the experience-vs-tenure check in 2.4 flags rows where stated experience is shorter than time at the company.

### 7.3 Limitations
- The flat salary-vs-experience and performance-by-department results, together with round percentages such as 60/40 for work mode, suggest the data was **randomly generated**. These findings describe this dataset and should not be generalized to real organisations.
- Salary averages in sections 5 and 7 use **capped** salaries (`Salary_INR`); the raw values are kept in `Salary_Original`.
- Because the dataset has only a snapshot of employees, causal claims (e.g. why people resign) cannot be made from it.